In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
class Order_app(TypedDict):
    user_query: str
    restaurant_name: str
    person_count: int
    time: str
    message: str
    avilable_status: bool
    booking_status: bool


# -------------------------
# 1. Create LLM
# -------------------------
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
load_dotenv()

model = init_chat_model(
    "openai:gpt-5.4-nano",
    temperature=0
)

In [ ]:
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv
load_dotenv()

model = init_chat_model(
    "openai:gpt-5.4-nano",
    temperature=0
)
query=" I want to book a table for 4 people at The Italian Bistro at 7 PM tonight."
prompt = f"""
Understand this restaurant booking request:

{query}

Extract the output in json format with the following fields:
- restaurant_name
- person_count
- time

"""

response = model.invoke(prompt)
   

In [ ]:

import json


text = response.content.replace("```json", "").replace("```", "").strip()

# Convert JSON string → Python dictionary
data = json.loads(text)
data

In [ ]:
restaurant_name = data.get("restaurant_name")
restaurant_name

In [ ]:
import json




def understand_request(state: Order_app): # node 1

    print("Understanding user request...")

    prompt = f"""
    Understand this restaurant booking request:

    {state["user_query"]}

    Extract the output in json format with the following fields:
    - restaurant_name
    - person_count
    - time

    """

    response = model.invoke(prompt)
    text = response.content.replace("```json", "").replace("```", "").strip()

    # Convert JSON string → Python dictionary
    data = json.loads(text)
    data

    return {
        "restaurant_name": data.get("restaurant_name"),
        "person_count": data.get("person_count"),
        "time": data.get("time")
    }

In [ ]:
def check_restaurant(state: Order_app): # node 2

    print("Checking restaurant...")

    return {
        "available_status": True
    }

In [ ]:

def book_table(state: Order_app): # node 3

    print("Booking table...")

    return {

        "booking_status": True
    }

In [ ]:
# ------------------------- 

def confirm_booking(state: Order_app): # node 4

    print("Confirming booking...")

    return {
        "message": "Your restaurant booking is confirmed!"
    }

In [ ]:

graph = StateGraph(Order_app)

In [ ]:
graph.add_node(
    "node-1-understand_request",
    understand_request
)
graph.add_node(
    "node-2-check_restaurant",
    check_restaurant
)
graph.add_node(
    "node-3-book_table",
    book_table
)
graph.add_node(
    "node-4-confirm_booking",
    confirm_booking
)

In [ ]:

graph.add_edge(
    START,
    "node-1-understand_request"
)
graph.add_edge(
    "node-1-understand_request",
    "node-2-check_restaurant"
)
graph.add_edge(
    "node-2-check_restaurant",
    "node-3-book_table"
)
graph.add_edge(
    "node-3-book_table",
    "node-4-confirm_booking"
)
graph.add_edge(
    "node-4-confirm_booking",
    END
)

In [ ]:

app = graph.compile()

In [ ]:
from IPython.display import Image, display

display(
    Image(
        app.get_graph().draw_mermaid_png()
    )
)

In [ ]:

result = app.invoke({"user_query": "Book an Hyderabadi restaurant for 8 people at 10 PM"})
{'user_query': 'Book an Hyderabadi restaurant for 8 people at 10 PM',
 'restaurant_name': 'Hyderabadi restaurant',
 'person_count': 8,
 'time': '10:00 PM',
 'message': 'Your restaurant booking is confirmed!',
 'booking_status': True}

In [ ]:
result

In [ ]:
# -------------------------
# 10. Invoke
# -------------------------

result = app.invoke({
    "user_query": "Book an Italian restaurant for 4 people at 8 PM",
    "restaurant": "",
    "people":"" ,
    "time": "",
    "message": ""
})


print(result)